# 02 - Validacao PyTorch

Recriacao da MLP com `nn.Linear` e comparacao das curvas com a implementacao NumPy.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve() / 'src'))

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from medmnist import PathMNIST
from utils import set_seed

set_seed(42)

In [ ]:
class TorchMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 9))
    def forward(self, x):
        return self.net(x)

train_ds = PathMNIST(split='train', size=28, download=True)
x = torch.tensor(train_ds.imgs, dtype=torch.float32).flatten(1) / 255.0
y = torch.tensor(train_ds.labels.reshape(-1), dtype=torch.long)
loader = DataLoader(TensorDataset(x, y), batch_size=128, shuffle=True)

model = TorchMLP()
opt = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)
criterion = nn.CrossEntropyLoss()
history_torch = []

for epoch in range(20):
    losses = []
    for xb, yb in loader:
        opt.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    with torch.no_grad():
        pred = model(x[:5000]).argmax(1)
        acc = (pred == y[:5000]).float().mean().item()
    history_torch.append({'epoch': epoch + 1, 'loss': sum(losses) / len(losses), 'acc_sample': acc})
    print(history_torch[-1])

Plote aqui as curvas NumPy e PyTorch sobrepostas e registre se a diferenca final de acuracia ficou em ate 2 pontos percentuais.